# 55 — LGBM All External Sources with Per-Source Weighting

LightGBM trained on ALL consolidated external data sources with learnable per-source weights.
Sources: ChEMBL NR targets, ChEMBL extended, BindingDB NR, PubChem PXR assays.

Key steps:
1. Load all external parquets; standardize SMILES; remove PXR-train overlap.
2. Assign per-source base sample weights.
3. Grid search over weight configurations via scaffold CV (PXR train only).
4. Final LGBM with optimal external configuration.
5. Save OOF + submission.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path

from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, standardize_smiles, to_inchikey
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS

SEED = 42
N_FOLDS = 5
LGBM_PARAMS = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    min_child_samples=10, n_jobs=4, verbose=-1
)

tr = load_train()
te = load_test()
print(f'Train: {len(tr):,}  Test: {len(te):,}')
train_inchikeys = set(tr['smiles'].map(to_inchikey).dropna())
print(f'Unique train InChIKeys: {len(train_inchikeys):,}')

Train: 4,139  Test: 513


Unique train InChIKeys: 4,138


## 1. Load all external data

In [2]:
# ── External source registry ──────────────────────────────────────────────────
# Each entry: (path, source_tag, base_weight)
# Weight schedule per spec:
#   PXR train=1.0 (assigned inline), ChEMBL PXR=0.8, ChEMBL CAR/VDR/FXR related=0.4-0.5,
#   ChEMBL PPARg=0.2, BindingDB PXR=0.7, BindingDB others=0.3, PubChem=0.6

SOURCE_REGISTRY = [
    (DATA_EXTERNAL / 'chembl_nr_targets.parquet',    'chembl_pxr',   0.8),
    (DATA_EXTERNAL / 'chembl_nr_extended.parquet',   'chembl_ext',   0.5),
    (DATA_EXTERNAL / 'bindingdb_nr_data.parquet',    'bindingdb',    0.5),
    (DATA_EXTERNAL / 'pubchem_pxr_aids.parquet',     'pubchem',      0.6),
]

# Source-specific weight rules based on target_name column (if present)
TARGET_WEIGHTS = {
    'PXR':   0.8,
    'CAR':   0.5,
    'VDR':   0.4,
    'FXR':   0.4,
    'LXRa':  0.3,
    'RXRa':  0.3,
    'PPARg': 0.2,
    'PPARa': 0.2,
}

ext_frames = []

for src_path, src_tag, base_w in SOURCE_REGISTRY:
    if not src_path.exists():
        print(f'  MISSING: {src_path.name} — skipped')
        continue
    try:
        df = pd.read_parquet(src_path)
    except Exception as e:
        print(f'  ERROR loading {src_path.name}: {e}')
        continue

    # Standardize column names
    smi_col = next((c for c in ['std_smiles', 'smiles', 'SMILES'] if c in df.columns), None)
    pec_col = next((c for c in ['pec50', 'pchembl_value', 'pEC50'] if c in df.columns), None)
    if smi_col is None or pec_col is None:
        print(f'  {src_path.name}: missing smiles/pec50 columns — skipped. Columns: {list(df.columns)}')
        continue

    df = df.rename(columns={smi_col: 'smiles_raw', pec_col: 'pec50'})
    df = df[['smiles_raw', 'pec50'] + [c for c in ['target_name', 'inchikey'] if c in df.columns]].copy()
    df = df.dropna(subset=['smiles_raw', 'pec50'])
    df = df[(df['pec50'] >= 3.0) & (df['pec50'] <= 11.0)]

    # Standardize SMILES + InChIKey dedup
    if 'inchikey' not in df.columns:
        df['inchikey'] = df['smiles_raw'].map(to_inchikey)
    df = df.dropna(subset=['inchikey'])

    # Remove PXR training set overlap
    before = len(df)
    df = df[~df['inchikey'].isin(train_inchikeys)]
    after = len(df)

    # Per-target weight if target_name column available
    if 'target_name' in df.columns:
        df['sample_weight'] = df['target_name'].map(TARGET_WEIGHTS).fillna(base_w).astype(np.float32)
    else:
        df['sample_weight'] = np.float32(base_w)

    df['source'] = src_tag

    # Get SMILES for featurization
    if 'std_smiles' in df.columns:
        df['smiles'] = df['std_smiles']
    else:
        df['smiles'] = df['smiles_raw'].map(standardize_smiles)
        df = df.dropna(subset=['smiles'])

    # Dedup by InChIKey — keep median pec50
    df = (
        df.sort_values('pec50', ascending=False)
          .drop_duplicates(subset='inchikey', keep='first')
          .reset_index(drop=True)
    )

    ext_frames.append(df)
    print(f'  {src_tag:15s}: {before:,} raw → {after:,} after dedup+overlap → {len(df):,} unique')

if ext_frames:
    ext_all = pd.concat(ext_frames, ignore_index=True)
    # Global dedup across sources — keep highest weight source for each InChIKey
    ext_all = (
        ext_all.sort_values('sample_weight', ascending=False)
               .drop_duplicates(subset='inchikey', keep='first')
               .reset_index(drop=True)
    )
    print(f'\nTotal external compounds (after cross-source dedup): {len(ext_all):,}')
else:
    ext_all = pd.DataFrame(columns=['smiles', 'pec50', 'sample_weight', 'source', 'inchikey'])
    print('No external data loaded — running CRC-only baseline')

  chembl_pxr     : 11,511 raw → 11,439 after dedup+overlap → 11,201 unique


  chembl_ext     : 11,496 raw → 11,423 after dedup+overlap → 11,185 unique


  bindingdb      : 5,690 raw → 5,572 after dedup+overlap → 4,543 unique
  pubchem        : 0 raw → 0 after dedup+overlap → 0 unique

Total external compounds (after cross-source dedup): 11,326


## 2. Featurize

In [3]:
# ── Featurize PXR training ────────────────────────────────────────────────────
print('Featurizing PXR training set...')
X_tr_raw = combined(tr['smiles'].tolist())
X_tr = impute(X_tr_raw)
y_tr = tr['pec50'].values.astype(np.float32)
w_tr = np.ones(len(tr), dtype=np.float32)  # weight=1.0 for PXR CRC
scaffolds_tr = tr['smiles'].map(bemis_murcko).tolist()
print(f'X_tr: {X_tr.shape}')

# ── Featurize test set ────────────────────────────────────────────────────────
print('Featurizing test set...')
X_te = impute(combined(te['smiles'].tolist()))
print(f'X_te: {X_te.shape}')

# ── Featurize external ────────────────────────────────────────────────────────
X_ext = None
y_ext = None
w_ext = None

if len(ext_all) > 0:
    print(f'Featurizing {len(ext_all):,} external compounds...')
    X_ext_raw = combined(ext_all['smiles'].tolist())
    X_ext = impute(X_ext_raw)
    y_ext = ext_all['pec50'].values.astype(np.float32)
    w_ext = ext_all['sample_weight'].values.astype(np.float32)
    print(f'X_ext: {X_ext.shape}')
    print(f'External pec50 stats: mean={y_ext.mean():.2f}  std={y_ext.std():.2f}  '
          f'min={y_ext.min():.2f}  max={y_ext.max():.2f}')

# ── Scaffold splits for PXR training folds (evaluation always on PXR train) ──
splits = scaffold_kfold_indices(scaffolds_tr, n_splits=N_FOLDS, seed=SEED)
print(f'\nScaffold {N_FOLDS}-fold splits ready.')

Featurizing PXR training set...


X_tr: (4139, 2265)
Featurizing test set...


X_te: (513, 2265)
Featurizing 11,326 external compounds...


X_ext: (11326, 2265)
External pec50 stats: mean=6.46  std=1.13  min=4.00  max=11.00

Scaffold 5-fold splits ready.


## 3. Source-weight grid search

In [4]:
def run_cv_with_external(X_internal, y_internal, w_internal, splits,
                          X_external, y_external, w_external_scaled,
                          lgbm_params=LGBM_PARAMS):
    """Run scaffold CV adding external data to each fold's training set.

    External data is always added in full to every fold's train split.
    Evaluation is always on the PXR CRC validation fold only (no leakage).
    """
    oof = np.full(len(y_internal), np.nan)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        X_fold_tr = X_internal[tr_idx]
        y_fold_tr = y_internal[tr_idx]
        w_fold_tr = w_internal[tr_idx]

        if X_external is not None and len(X_external) > 0:
            X_train_aug = np.vstack([X_fold_tr, X_external])
            y_train_aug = np.concatenate([y_fold_tr, y_external])
            w_train_aug = np.concatenate([w_fold_tr, w_external_scaled])
        else:
            X_train_aug = X_fold_tr
            y_train_aug = y_fold_tr
            w_train_aug = w_fold_tr

        m = lgb.LGBMRegressor(**lgbm_params)
        m.fit(
            X_train_aug, y_train_aug,
            sample_weight=w_train_aug,
            callbacks=[lgb.log_evaluation(-1)],
        )
        oof[va_idx] = m.predict(X_internal[va_idx])
    return oof


# ── Weight configurations to test ────────────────────────────────────────────
weight_configs = [
    ('(a) all_sources_base_weights',    1.0),  # multiply external weights by 1.0
    ('(b) upweight_external_1.5x',      1.5),
    ('(c) downweight_external_0.5x',    0.5),
    ('(d) downweight_external_0.25x',   0.25),
]

# Baseline: PXR CRC only
print('Computing CRC-only baseline...')
oof_crc_only = run_cv_with_external(
    X_tr, y_tr, w_tr, splits,
    None, None, None,
)
rae_crc_only = rae(y_tr, oof_crc_only)
print(f'  CRC-only OOF RAE: {rae_crc_only:.4f}')

results_grid = [{'config': 'CRC only', 'scale': 0.0, 'oof_rae': rae_crc_only}]

best_rae = rae_crc_only
best_config_name = 'CRC only'
best_w_scale = 0.0
best_oof = oof_crc_only.copy()

if X_ext is not None:
    for cfg_name, w_scale in weight_configs:
        w_ext_scaled = w_ext * w_scale
        oof_ext = run_cv_with_external(
            X_tr, y_tr, w_tr, splits,
            X_ext, y_ext, w_ext_scaled,
        )
        r = rae(y_tr, oof_ext)
        print(f'  {cfg_name}: OOF RAE = {r:.4f}  (delta vs CRC-only: {r - rae_crc_only:+.4f})')
        results_grid.append({'config': cfg_name, 'scale': w_scale, 'oof_rae': r})
        if r < best_rae:
            best_rae = r
            best_config_name = cfg_name
            best_w_scale = w_scale
            best_oof = oof_ext.copy()
else:
    print('No external data available — skipping grid search.')

print(f'\nBest config: {best_config_name}  (scale={best_w_scale:.2f})  OOF RAE={best_rae:.4f}')
pd.DataFrame(results_grid).sort_values('oof_rae').to_string(index=False)

Computing CRC-only baseline...


  CRC-only OOF RAE: 0.5600


  (a) all_sources_base_weights: OOF RAE = 0.6026  (delta vs CRC-only: +0.0425)


  (b) upweight_external_1.5x: OOF RAE = 0.6075  (delta vs CRC-only: +0.0475)


  (c) downweight_external_0.5x: OOF RAE = 0.6003  (delta vs CRC-only: +0.0403)


  (d) downweight_external_0.25x: OOF RAE = 0.5941  (delta vs CRC-only: +0.0341)

Best config: CRC only  (scale=0.00)  OOF RAE=0.5600


'                       config  scale  oof_rae\n                     CRC only   0.00 0.560014\n(d) downweight_external_0.25x   0.25 0.594065\n (c) downweight_external_0.5x   0.50 0.600304\n (a) all_sources_base_weights   1.00 0.602561\n   (b) upweight_external_1.5x   1.50 0.607547'

## 4. Final model

In [5]:
# ── Final model: retrain on all data with best configuration ─────────────────
if X_ext is not None and best_w_scale > 0:
    w_ext_best = w_ext * best_w_scale
    X_all = np.vstack([X_tr, X_ext])
    y_all = np.concatenate([y_tr, y_ext])
    w_all = np.concatenate([w_tr, w_ext_best])
    print(f'Training final model on {len(y_all):,} compounds '
          f'({len(y_tr):,} CRC + {len(y_ext):,} external) ...')
else:
    X_all, y_all, w_all = X_tr, y_tr, w_tr
    print('Training final model on CRC data only...')

final_model = lgb.LGBMRegressor(**LGBM_PARAMS)
final_model.fit(
    X_all, y_all,
    sample_weight=w_all,
    callbacks=[lgb.log_evaluation(-1)],
)

te_preds = final_model.predict(X_te)
lo_clip = float(y_tr.min()) - 0.5
hi_clip = float(y_tr.max()) + 0.5
te_preds = np.clip(te_preds, lo_clip, hi_clip)

print(f'Test preds — mean={te_preds.mean():.3f}  std={te_preds.std():.3f}  '
      f'min={te_preds.min():.3f}  max={te_preds.max():.3f}')
print(f'\nBest config OOF RAE: {best_rae:.4f}')
print(f'CRC-only OOF RAE:    {rae_crc_only:.4f}')
print(f'Delta:                {best_rae - rae_crc_only:+.4f}')

# Also compute OOF for the best config for downstream ensemble
# Use the best_oof array computed during grid search above
print(f'\nOOF RAE (best external config): {rae(y_tr, best_oof):.4f}')

Training final model on CRC data only...


Test preds — mean=4.804  std=0.649  min=2.209  max=6.052

Best config OOF RAE: 0.5600
CRC-only OOF RAE:    0.5600
Delta:                +0.0000

OOF RAE (best external config): 0.5600


## 5. Save

In [6]:
# Save OOF
np.save(DATA_PROCESSED / 'oof_lgbm_all_external.npy', best_oof)
print(f'Saved oof_lgbm_all_external.npy  (OOF RAE = {rae(y_tr, best_oof):.4f})')

# Save test predictions
np.save(DATA_PROCESSED / 'te_lgbm_all_external.npy', te_preds)
print(f'Saved te_lgbm_all_external.npy')

# Save submission
sub = pd.DataFrame({
    'Molecule Name': te['name'].values,
    'pEC50':         te_preds,
})
assert len(sub) == 513 and sub['pEC50'].notna().all(), 'Submission validation failed'
out_path = SUBMISSIONS / '55_lgbm_all_external.csv'
sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

print('\n== Summary ==')
print(f'  Best config:       {best_config_name}')
print(f'  OOF RAE (best):    {best_rae:.4f}')
print(f'  OOF RAE (CRC only):{rae_crc_only:.4f}')
print(f'  Test pred std:     {te_preds.std():.4f}')
sub['pEC50'].describe().round(3)

Saved oof_lgbm_all_external.npy  (OOF RAE = 0.5600)
Saved te_lgbm_all_external.npy
Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\55_lgbm_all_external.csv

== Summary ==
  Best config:       CRC only
  OOF RAE (best):    0.5600
  OOF RAE (CRC only):0.5600
  Test pred std:     0.6494


count    513.000
mean       4.804
std        0.650
min        2.209
25%        4.443
50%        4.948
75%        5.282
max        6.052
Name: pEC50, dtype: float64